In [1]:
def calculate_iou(boxA, boxB):
    # 解包坐标 x1,y1,x2,y2
    x1a, y1a, x2a, y2a = boxA
    x1b, y1b, x2b, y2b = boxB

    # 交集左上角、右下角
    x1_inter = max(x1a, x1b)
    y1_inter = max(y1a, y1b)
    x2_inter = min(x2a, x2b)
    y2_inter = min(y1a, y2b)

    # 无交集则面积为0
    w_inter = max(0, x2_inter - x1_inter)
    h_inter = max(0, y2_inter - y1_inter)
    area_inter = w_inter * h_inter

    # 两个框各自面积
    areaA = (x2a - x1a) * (y2a - y1a)
    areaB = (x2b - x1b) * (y2b - y1b)

    # 并集
    area_union = areaA + areaB - area_inter
    iou = area_inter / area_union
    return iou, area_inter, area_union


# 给定边框
A = [10, 10, 50, 50]
B = [30, 30, 70, 70]
iou, inter, union = calculate_iou(A, B)

print(f"交集面积 = {inter}")
print(f"并集面积 = {union}")
print(f"IoU = {iou:.6f}")
print(f"IoU 分数精确分数 = 1/7 ≈ {1/7:.6f}")

交集面积 = 0
并集面积 = 3200
IoU = 0.000000
IoU 分数精确分数 = 1/7 ≈ 0.142857


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, eps=0.1, K=None):
    """
    标签平滑交叉熵损失
    :param logits: 模型原始输出，shape [N, K]，未过softmax
    :param labels: 真实标签索引，shape [N]，int类型
    :param eps: 平滑系数ε
    :param K: 总类别数，不传入自动从logits获取
    :return: 平均损失标量
    """
    N, K = logits.shape
    # 原始softmax概率
    prob = F.softmax(logits, dim=1)

    # 构造平滑后的标签分布
    y_smooth = torch.full_like(prob, fill_value=eps / (K - 1))
    # 真实类别位置赋值 1-ε
    y_smooth.scatter_(dim=1, index=labels.unsqueeze(1), value=1. - eps)

    # 逐样本交叉熵
    loss = -torch.sum(y_smooth * torch.log(prob + 1e-8), dim=1)
    return torch.mean(loss)


# ---------------- 测试代码 ----------------
if __name__ == "__main__":
    # 批次N=4，类别K=5
    batch_logits = torch.randn(4, 5)
    batch_labels = torch.tensor([0, 2, 1, 4])

    loss_smooth = label_smoothing_cross_entropy(batch_logits, batch_labels, eps=0.1)
    print(f"标签平滑(ε=0.1)交叉熵损失值 = {loss_smooth.item():.4f}")

    # 对比原生nn.CrossEntropyLoss(等价ε=0)
    loss_ori = F.cross_entropy(batch_logits, batch_labels)
    print(f"原生独热编码交叉熵损失值 = {loss_ori.item():.4f}")

标签平滑(ε=0.1)交叉熵损失值 = 1.8904
原生独热编码交叉熵损失值 = 1.8544
